# LAB 01 — PSO para Balanceamento Dinâmico de Carga em Datacenters

Otimização de um vetor de pesos contínuos para 6 zonas de disponibilidade (AZs), minimizando a temperatura média ponderada e aplicando penalidade quando alguma AZ ultrapassa 75 °C.


### Cenário

Coeficientes de aquecimento das 6 AZs:

`C = [42, 35, 58, 30, 50, 65] °C`

O vetor de decisão é `W = [w1, ..., w6]`, com `wi >= 0` e `sum(W) = 1.0`.

A temperatura de cada AZ é modelada como `T_i = C_i * (1 + α * w_i)`, onde `α` representa o efeito do carregamento. A função objetivo minimiza a temperatura média ponderada `sum(w_i * T_i)` e aplica penalidade externa se qualquer `T_i > 75 °C`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# Dados do problema
C = np.array([42.0, 35.0, 58.0, 30.0, 50.0, 65.0])
LIMITE_TEMPERATURA = 75.0
ALPHA = 0.5

# Parâmetros do PSO
N_ITER = 100
W_INERCIA = 0.7
C1 = 1.5
C2 = 1.5
V_MAX = 0.20

def normalizar_posicao(posicao):
    """Garante pesos não negativos e soma exatamente igual a 1."""
    posicao = np.maximum(posicao, 0.0)
    soma = np.sum(posicao)
    if soma == 0:
        return np.ones(len(posicao)) / len(posicao)
    return posicao / soma

def calcular_temperaturas(W):
    """Calcula a temperatura de cada AZ considerando o carregamento."""
    return C * (1 + ALPHA * W)

def fitness(W):
    """Minimiza a temperatura média ponderada + penalidade externa."""
    W = normalizar_posicao(W)
    temperaturas = calcular_temperaturas(W)
    temperatura_media = np.sum(W * temperaturas)

    excesso = np.maximum(temperaturas - LIMITE_TEMPERATURA, 0)
    penalidade = 1000 * np.sum(excesso)

    return temperatura_media + penalidade


## Implementação do PSO contínuo

A implementação abaixo mantém a posição, velocidade, `P_best`, `G_best` e o histórico de evolução do melhor fitness. Após cada atualização de posição, o vetor é normalizado para garantir `sum(wi) = 1.0`.

In [ ]:
class PSO:
    def __init__(self, n_particulas, n_dimensoes=6, n_iter=100):
        self.n_particulas = n_particulas
        self.n_dimensoes = n_dimensoes
        self.n_iter = n_iter

        # Inicialização das posições como distribuições válidas
        self.posicoes = np.random.dirichlet(np.ones(n_dimensoes), size=n_particulas)
        self.velocidades = np.random.uniform(-0.05, 0.05, size=(n_particulas, n_dimensoes))

        self.p_best = self.posicoes.copy()
        self.p_best_fitness = np.array([fitness(p) for p in self.p_best])

        melhor_idx = np.argmin(self.p_best_fitness)
        self.g_best = self.p_best[melhor_idx].copy()
        self.g_best_fitness = self.p_best_fitness[melhor_idx]

        self.historico = [self.g_best_fitness]

    def executar(self):
        for _ in range(self.n_iter):
            r1 = np.random.rand(self.n_particulas, self.n_dimensoes)
            r2 = np.random.rand(self.n_particulas, self.n_dimensoes)

            # Atualização de velocidade
            self.velocidades = (
                W_INERCIA * self.velocidades
                + C1 * r1 * (self.p_best - self.posicoes)
                + C2 * r2 * (self.g_best - self.posicoes)
            )

            self.velocidades = np.clip(self.velocidades, -V_MAX, V_MAX)

            # Atualização de posição
            self.posicoes += self.velocidades

            # Operador de normalização obrigatório
            for i in range(self.n_particulas):
                self.posicoes[i] = normalizar_posicao(self.posicoes[i])

            # Avaliação e atualização de P_best
            fitness_atual = np.array([fitness(p) for p in self.posicoes])
            melhorou = fitness_atual < self.p_best_fitness
            self.p_best[melhorou] = self.posicoes[melhorou]
            self.p_best_fitness[melhorou] = fitness_atual[melhorou]

            # Atualização de G_best
            melhor_idx = np.argmin(self.p_best_fitness)
            if self.p_best_fitness[melhor_idx] < self.g_best_fitness:
                self.g_best = self.p_best[melhor_idx].copy()
                self.g_best_fitness = self.p_best_fitness[melhor_idx]

            self.historico.append(self.g_best_fitness)

        return self.g_best, self.g_best_fitness, self.historico


## Execução para 10, 30 e 50 partículas

In [ ]:
tamanhos = [10, 30, 50]
resultados = {}

for n in tamanhos:
    # Seed diferente para cada experimento, mantendo reprodutibilidade
    np.random.seed(42 + n)
    pso = PSO(n_particulas=n, n_dimensoes=6, n_iter=N_ITER)
    melhor_W, melhor_fitness, historico = pso.executar()
    resultados[n] = {
        'W': melhor_W,
        'fitness': melhor_fitness,
        'historico': historico
    }

    print(f'População: {n}')
    print('Melhor W:', np.round(melhor_W, 6))
    print('Soma dos pesos:', np.sum(melhor_W))
    print('Fitness:', round(melhor_fitness, 6))
    print('Temperaturas:', np.round(calcular_temperaturas(melhor_W), 4))
    print('-' * 60)


## Evolução do fitness

O gráfico compara o melhor fitness global encontrado ao longo das iterações para as três populações.

In [ ]:
plt.figure(figsize=(10, 6))

for n in tamanhos:
    plt.plot(resultados[n]['historico'], label=f'{n} partículas')

plt.title('Evolução do melhor fitness — PSO')
plt.xlabel('Iteração')
plt.ylabel('Fitness (menor é melhor)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Tabela final dos experimentos

In [ ]:
linhas = []

for n in tamanhos:
    W = resultados[n]['W']
    temperaturas = calcular_temperaturas(W)

    linhas.append({
        'Partículas': n,
        'w1': W[0],
        'w2': W[1],
        'w3': W[2],
        'w4': W[3],
        'w5': W[4],
        'w6': W[5],
        'Soma W': np.sum(W),
        'Fitness': resultados[n]['fitness'],
        'Temp. Máxima': np.max(temperaturas)
    })

tabela = pd.DataFrame(linhas)
display(tabela.round(6))


## Validação das restrições

A validação confirma que os pesos são não negativos, somam 1.0 e que nenhuma temperatura final ultrapassa o limite crítico de 75 °C.

In [ ]:
for n in tamanhos:
    W = resultados[n]['W']
    temperaturas = calcular_temperaturas(W)

    soma_valida = np.isclose(np.sum(W), 1.0)
    pesos_validos = np.all(W >= 0)
    temperatura_valida = np.all(temperaturas <= LIMITE_TEMPERATURA)

    print(f'População {n}:')
    print(f'  Soma(W) = {np.sum(W):.12f} -> {soma_valida}')
    print(f'  Pesos >= 0 -> {pesos_validos}')
    print(f'  Todas as temperaturas <= {LIMITE_TEMPERATURA} °C -> {temperatura_valida}')
    print()


## Análise técnica

- O PSO utiliza uma representação contínua, adequada para os pesos de distribuição de tráfego.
- A normalização após cada atualização garante que a solução permaneça no espaço de distribuições com soma igual a 1.
- O `P_best` registra a melhor posição encontrada individualmente por cada partícula, enquanto o `G_best` registra a melhor solução encontrada por todo o enxame.
- A penalidade externa aumenta o fitness quando alguma temperatura ultrapassa 75 °C, tornando essas soluções menos atrativas para o processo de otimização.
- As três populações permitem observar o efeito do tamanho do enxame na velocidade de convergência e na qualidade da solução encontrada.

**Observação:** os valores numéricos apresentados pelo notebook são gerados pela execução do código no ambiente do Google Colab. O `seed` foi definido para permitir a reprodução dos resultados.